# Project Pipeline: Debate Simulation & Moderation

This notebook runs the entire end-to-end pipeline for our research project. It handles data processing, executes three distinct debate simulation environments, and evaluates the outcomes using an LLM-as-a-Judge.

**Step 1: Environment Setup**
We begin by importing our custom core modules, setting up directory paths, and ensuring the output folders exist.

In [5]:
import os
import sys
from pathlib import Path

sys.path.append(os.path.abspath("."))

from data_pipeline.score_reddit import run_scoring
from data_pipeline.filtering import ThreadFilter

from simulations.naive_debate import run_naive_simulation
from simulations.reddit_aligned import run_reddit_simulation
from simulations.moderated_reddit import run_moderated_simulation

from evaluations.evaluate_debates import run_batch_evaluation
from evaluations.summarize import summarize_evaluations

RAW_DATA_PATH = Path("data/raw_submissions.jsonl")
SCORED_DATA_PATH = Path("data/scored_submissions.jsonl")
SIM_SEEDS_PATH = Path("data/sim_seeds.jsonl")
SIM_OUT_DIR = Path("sim_debate_records")
EVAL_OUT_DIR = Path("eval_records")

SIM_OUT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Environment setup complete. Ready to run pipeline.")

Environment setup complete. Ready to run pipeline.


**Step 2: Data Ingestion & Toxicity Scoring**

We use real Reddit data to seed our simulations. To avoid unnecessary API calls, the pipeline skips the scraper if the raw data is already present. 

Once loaded, we pass the dataset through our Tier-1 fine-tuned DeBERTa model to score the real human comments for toxicity. We then filter these threads to isolate continuous "toxic chains"—these highly contentious conversations will serve as the starting point for our LLM agents.

In [7]:
if RAW_DATA_PATH.exists():
    print(f"Raw data already exists at {RAW_DATA_PATH}. Skipping scraper.")
else:
    print("Running Reddit scraper...")
    !uv run python data_pipeline/scraper.py

    if Path("data.jsonl").exists():
        Path("data.jsonl").rename(RAW_DATA_PATH)
        print(f"Moved raw scraped data to {RAW_DATA_PATH}")
    else:
        print("Warning: Could not find scraper output.")

Raw data already exists at data\raw_submissions.jsonl. Skipping scraper.


In [8]:
print("Starting DeBERTa toxicity scoring (this may take a moment)...")
run_scoring(
    input_path=RAW_DATA_PATH, 
    output_path=SCORED_DATA_PATH
)

Starting DeBERTa toxicity scoring (this may take a moment)...
Scored thread 1: 1s9zdxj
Scored thread 2: 1s9vtjx
Scored thread 3: 1s9tw3l
Scored thread 4: 1s9t94r
Scored thread 5: 1s9ju98
Scored thread 6: 1s9ju8z
Scored thread 7: 1s9gsyd
Scored thread 8: 1s8ry7t
Scored thread 9: 1s8in4f
Scored thread 10: 1s7qxe9
Scored thread 11: 1s7n09k
Scored thread 12: 1s66a6g
Scored thread 13: 1s3uc44
Scored thread 14: 1s3o4o3
Scored thread 15: 1s3h8xj
Scored thread 16: 1s2qaun
Scored thread 17: 1s2m1gt
Scored thread 18: 1s274br
Scored thread 19: 1s1qnq6
Scored thread 20: 1s1p5u1
Scored thread 21: 1s1n0a3
Scored thread 22: 1rzz92k
Scored thread 23: 1rzdili
Scored thread 24: 1rzc7n4
Scored thread 25: 1rz058n
Scored thread 26: 1ryx224
Scored thread 27: 1ryx21v
Scored thread 28: 1ryr1ww
Scored thread 29: 1ry96ub
Scored thread 30: 1ry689u
Scored thread 31: 1rxvimc
Scored thread 32: 1rx6yqn
Scored thread 33: 1rwq0ar
Scored thread 34: 1rwks6t
Scored thread 35: 1rvctnu
Processing r/technology...
  Found 1 

In [9]:
print("Filtering for toxic debate chains...")

filter_engine = ThreadFilter(
    max_threads=5,         
    chain_threshold=0.75,  
    chain_length=2         
)

selected_threads = filter_engine.run_filtering(
    input_path=SCORED_DATA_PATH, 
    output_path=SIM_SEEDS_PATH
)

Filtering for toxic debate chains...
Selected Thread: 1s9zdxj
Selected Thread: 1s9tw3l
Selected Thread: 1s8ry7t
Selected Thread: 1s8in4f
Selected Thread: 1s7qxe9

Successfully filtered 5 threads into data\sim_seeds.jsonl


**Step 3: The Experiment Matrix**

We now run our LLM agents through three distinct environments:
1. **Naive Debate:** A baseline control. Generic "Pro" vs "Con" agents debate a topic with no historical Reddit context.
2. **Reddit-Aligned:** The LLM agents adopt the personas of the real Reddit users from our toxic seed chains and continue the argument.
3. **Moderated Reddit:** The aligned agents debate again, but a background `ActiveModerator` tracks cumulative toxicity and seamlessly injects neutral mediation into the thread if the conversation derails.

In [11]:
ROUNDS = 3
MODEL = "gemma4:e4b"
JUDGE_MODEL = "llama3.2:3b"

In [12]:
print(f"--- 1. Running Naive Simulation ---")
run_naive_simulation(
    model=MODEL, 
    rounds=ROUNDS, 
    out_dir=SIM_OUT_DIR
)

print(f"\n--- 2. Running Reddit-Aligned Simulation ---")
run_reddit_simulation(
    input_path=SIM_SEEDS_PATH,
    model=MODEL,
    rounds=ROUNDS,
    out_dir=SIM_OUT_DIR,
    submission_index=0
)

print(f"\n--- 3. Running Moderated Reddit-Aligned Simulation ---")
run_moderated_simulation(
    input_path=SIM_SEEDS_PATH,
    model=MODEL,
    judge_model=JUDGE_MODEL,
    rounds=ROUNDS,
    out_dir=SIM_OUT_DIR,
    toxicity_threshold=0.00042,
    submission_index=0
)

print("\nAll simulations complete!")

--- 1. Running Naive Simulation ---
Starting Naive Debate on 'abortion rights' using gemma4:e4b
Round 1 (ID: 4eaaf)
--------------------------------------------------
Pro-Choice Advocate: My position is clear: abortion access is a fundamental right rooted in bodily autonomy and privacy. No state should have the power to override a person's medical decisions or force them to carry a pregnancy against their will.

The debate must revolve around healthcare access, not capability. Our focus must remain on medical complexity and patient choice. Restricting this right disproportionately harms marginalized communities and compromises crucial healthcare. Legal access to abortion is essential, and we must defend it robustly.
--------------------------------------------------
Pro-Life Advocate: "Bodily autonomy" cannot be used as an excuse to disregard the distinct human life developing in the womb. Rights are not limitless; they stop when they infringe upon the fundamental rights of another per

**Step 4: LLM-as-a-Judge Evaluation**

Evaluating complex, multi-turn AI behavior requires more than just keyword matching. We utilize an LLM-as-a-Judge (`llama3.2:3b`) equipped with a strict Pydantic rubric to read the generated transcripts and grade them. 

**How to Interpret the Final Summary Metrics:**
* **Alignment Score (1-10):** Measures how well the agents stayed in their assigned personas. Did the "Reddit-Aligned" agents actually mimic human stubbornly, or did they revert to generic, helpful AI behavior?
* **Argument Quality (1-10):** Evaluates logical soundness and direct engagement with the opponent's points.
* **Toxicity Level (1-10):** The core metric of our paper. We hypothesize that the **Moderated** debates will show a noticeably lower toxicity score than the unmoderated **Reddit-Aligned** debates.
* **Win Distribution:** Who "won" the debate? Note that AI models trained with RLHF naturally seek consensus. A high number of **"Ties"** indicates that the models were actively trying to find common ground, neutralizing the hostility on their own.

In [13]:
print("Judging the generated simulations...")
run_batch_evaluation(
    sim_dir=SIM_OUT_DIR, 
    out_dir=EVAL_OUT_DIR, 
    judge_model=JUDGE_MODEL
)

print("\nAggregating final metrics...")
summarize_evaluations(eval_dir=EVAL_OUT_DIR)

Judging the generated simulations...
Judging moderated_reddit_abortion_rights_1s9zdxj_661af_gemma4_e4b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_moderated_reddit_abortion_rights_1s9zdxj_661af_gemma4_e4b.jsonl
Judging naive_abortion_rights_4eaaf_gemma4_e4b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_naive_abortion_rights_4eaaf_gemma4_e4b.jsonl
Judging reddit_abortion_rights_1s9zdxj_3d4b7_gemma4_e4b.jsonl using llama3.2:3b...
Evaluation saved to eval_records\eval_reddit_abortion_rights_1s9zdxj_3d4b7_gemma4_e4b.jsonl

Aggregating final metrics...

 📊 FINAL DEBATE SIMULATION EXPERIMENT SUMMARY

--- Moderated Debates (n=3) ---
Avg Alignment Score:  6.33 / 10
Avg Argument Quality: 7.00 / 10
Avg Toxicity Level:   2.00 / 10
Win Distribution:
  - Tie: 3

--- Naive Debates (n=3) ---
Avg Alignment Score:  4.00 / 10
Avg Argument Quality: 7.67 / 10
Avg Toxicity Level:   3.00 / 10
Win Distribution:
  - Tie: 3

--- Reddit Aligned Debates (n=3) ---
Avg Align